## 数据校验 Pydantic


### 一、Pydantic是什么
- 定义：Python 最流行的`数据校验与类型转换库`
- 核心思想：用 `类型注解（Type Hints）` 定义数据结构，自动校验
- 典型场景：API 请求/响应校验、配置管理、数据清洗、FastAPI 底层依赖
- 当前主流版本：`Pydantic V2`（2023年发布，性能提升 5~50 倍）

安装
```
# 安装
pip install pydantic
# 如需邮件/URL等额外校验
pip install pydantic[email]
```

🚀 语法速查卡片

In [28]:
# ========== 定义模型 ==========
from pydantic import BaseModel, Field, field_validator, model_validator, ConfigDict
from typing import Optional, List, Literal

class MyModel(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True, extra="forbid")

    name: str = Field(..., min_length=1, max_length=50)
    age: int = Field(default=0, ge=0, le=150)
    email: Optional[str] = None
    tags: List[str] = []
    status: Literal["on", "off"] = "off"

    @field_validator("name")
    @classmethod
    def check_name(cls, v):
        if not v.strip():
            raise ValueError("不能为空")
        return v

# ========== 创建实例 ==========
obj = MyModel(name="Alice", age=25)

# ========== 序列化 ==========
obj.model_dump()              # → dict
obj.model_dump_json()         # → str (JSON)
obj.model_dump(by_alias=True) # 使用别名

# ========== 反序列化 ==========
MyModel.model_validate({"name": "Bob", "age": 30})
MyModel.model_validate_json('{"name":"Bob","age":30}')

# ========== 错误处理 ==========
from pydantic import ValidationError
try:
    MyModel(name="", age=-1)
except ValidationError as e:
    print(e.errors())

# ========== 更新字段 ==========
obj2 = obj.model_copy(update={"age": 26})

# ========== JSON Schema ==========
MyModel.model_json_schema()


[{'type': 'string_too_short', 'loc': ('name',), 'msg': 'String should have at least 1 character', 'input': '', 'ctx': {'min_length': 1}, 'url': 'https://errors.pydantic.dev/2.11/v/string_too_short'}, {'type': 'greater_than_equal', 'loc': ('age',), 'msg': 'Input should be greater than or equal to 0', 'input': -1, 'ctx': {'ge': 0}, 'url': 'https://errors.pydantic.dev/2.11/v/greater_than_equal'}]


{'additionalProperties': False,
 'properties': {'name': {'maxLength': 50,
   'minLength': 1,
   'title': 'Name',
   'type': 'string'},
  'age': {'default': 0,
   'maximum': 150,
   'minimum': 0,
   'title': 'Age',
   'type': 'integer'},
  'email': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Email'},
  'tags': {'default': [],
   'items': {'type': 'string'},
   'title': 'Tags',
   'type': 'array'},
  'status': {'default': 'off',
   'enum': ['on', 'off'],
   'title': 'Status',
   'type': 'string'}},
 'required': ['name'],
 'title': 'MyModel',
 'type': 'object'}

### 二、Pydantic 的核心功能

### 1. 模型定义与基础字段

#####  1.1 字段类型（Type Hints）

In [3]:
from typing import Optional, List, Dict, Union, Literal
from datetime import datetime, date
from pydantic import BaseModel, EmailStr, HttpUrl

class Demo(BaseModel):
    # 基础类型
    name: str
    age: int
    score: float
    is_active: bool

    # 可选字段（可为 None）
    nickname: str | None = None # Optional[str] = None

    # 列表
    tags: list[str] = [] # List[str] = []

    # 字典
    metadata: dict[str, str] = {} # Dict[str, str] = {}

    # 联合类型
    id: int | str # Union[int, str]

    # 字面量（限定取值）
    status: Literal["active", "inactive", "banned"]

    # 日期时间
    created_at: datetime
    birthday: date

    # 特殊类型（需安装 pydantic[email]）
    email: EmailStr
    website: HttpUrl

> ⚠️ 规则：没有默认值的字段 = 必填字段

#####  1.2 默认值与必填

In [12]:
from pydantic import BaseModel
class Config(BaseModel):
    host: str                          # 必填（无默认值）
    port: int = 8080                   # 可选，默认 8080
    debug: bool = False                # 可选，默认 False
    name: str | None = None            # 可选，默认 None

config = Config(host="localhost", name = "123")
config

Config(host='localhost', port=8080, debug=False, name='123')

##### 1.3 Field — 字段约束与元数据
| 类别 | 参数 | 含义 | 示例 |
|---|---|---|---|
| **数值** | `gt` | 大于 | `Field(gt=0)` |
| | `ge` | 大于等于 | `Field(ge=0)` |
| | `lt` | 小于 | `Field(lt=100)` |
| | `le` | 小于等于 | `Field(le=99999)` |
| | `multiple_of` | 必须是某数的倍数 | `Field(multiple_of=0.5)` |
| **字符串** | `min_length` | 最小长度 | `Field(min_length=1)` |
| | `max_length` | 最大长度 | `Field(max_length=100)` |
| | `pattern` | 正则匹配 | `Field(pattern=r"^[a-z]+$")` |
| **列表** | `min_length` | 最少元素数 | `Field(min_length=1)` |
| | `max_length` | 最多元素数 | `Field(max_length=5)` |
| **通用** | `...` | 显式标记必填 | `Field(...)` |
| | `default` | 默认值（有它就变可选） | `Field(default="hello")` |
| | `alias` | 序列化 / 反序列化别名 | `Field(alias="displayName")` |
| | `frozen` | 冻结字段，创建后不可修改 | `Field(frozen=True)` |
| **元数据** | `title` | 字段标题 | `Field(title="用户名")` |
| | `description` | 字段描述 | `Field(description="仅内部使用")` |
| | `examples` | 示例值列表 | `Field(examples=["test"])` |
| | `json_schema_extra` | 任意额外 JSON Schema 信息 | `Field(json_schema_extra={"deprecated": True})` |

In [ ]:
from pydantic import BaseModel, Field

class Product(BaseModel):
    name: str = Field(
        ...,                          # ... 表示必填
        min_length=1,                 # 最小长度
        max_length=100,               # 最大长度
        description="商品名称",        # 描述（用于文档）
        examples=["iPhone 15"],       # 示例
    )
    price: float = Field(
        gt=0,                         # 大于 0
        le=999999,                    # 小于等于
        description="价格"
    )
    quantity: int = Field(
        default=1,                    # 默认值
        ge=1,                         # 大于等于 1
        le=10000                      # 小于等于 10000
    )


#### 2. 自定义验证器


| 知识点 | 功能 | 语法                                                                                                                                                                                               |
|---|---|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `field_validator` | 对单个字段做自定义校验、清洗、转换 | `@field_validator("字段名", mode="after")`<br>`@classmethod`<br>`def check(cls, v): `<br>`...`<br>`return v`                                                                                        |
| `model_validator` | 对整个模型做校验，常用于跨字段校验、模型级预处理 | `@model_validator(mode="after")`<br>`def check(self): `<br>`...` <br>`reurn self`<br><br>`@model_validator(mode="before")`<br>`@classmethod`<br>`def preprocess(cls, data): `<br>`...` <br> `return data` |

| 知识点 | 功能 | 模式 |
|---|---|---|
| `field_validator` | 字段级校验，作用于指定字段 | `mode="after"`：默认，字段类型转换后校验<br>`mode="before"`：字段类型转换前校验<br>`mode="wrap"`：包裹默认字段校验流程<br>`mode="plain"`：完全接管字段校验 |
| `model_validator` | 模型级校验，作用于整个模型 | `mode="after"`：所有字段校验通过后执行，接收模型实例<br>`mode="before"`：字段校验前执行，接收原始数据 |

In [16]:
from pydantic import BaseModel, field_validator, model_validator, ValidationInfo

class User(BaseModel):
    name: str
    age: int
    password: str
    confirm: str

    # ── field_validator：单字段 / 利用 info.data 读已验证字段 ──
    @field_validator("name")
    @classmethod
    def name_not_empty(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("名称不能为空")
        return v.strip()

    @field_validator("confirm")
    @classmethod
    def passwords_match(cls, v: str, info: ValidationInfo) -> str:
        if v != info.data.get("password"):
            raise ValueError("两次密码不一致")
        return v

    # ── model_validator：跨字段后处理 ──

    @model_validator(mode="after")
    def normalize_name(self):
        self.name = self.name.title()
        return self

User(name="alice", age=20, password="123", confirm="123")
# name='Alice'  ← model_validator 自动首字母大写

# User(password="abc", confirm="xyz") # ValueError: 两次密码不一致  ← field_validator 拦截

User(name='Alice', age=20, password='123', confirm='123')

#### 3. 模型方法（序列化、反序列化）

In [27]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int

user = User(name="Alice", age=25)
print("user:", user)

# ① 转为字典
d = user.model_dump() # {'name': 'Alice', 'age': 25}
print(type(d))

# ② 转为 JSON 字符串
j = user.model_dump_json() # '{"name":"Alice","age":25}'
print(type(j))

# ③ 从字典创建
user2 = User.model_validate({"name": "Bob", "age": 30})
print("user2:", user2)
# ④ 从 JSON 字符串创建
user3 = User.model_validate_json('{"name":"Charlie","age":35}')
print("user3", user3)

# ⑤ 获取 JSON Schema
schema = User.model_json_schema()

user: name='Alice' age=25
<class 'dict'>
<class 'str'>
user2: name='Bob' age=30
user3 name='Charlie' age=35


从 ORM 对象转换（最关键）

In [ ]:
class UserOut(BaseModel):
    model_config = ConfigDict(from_attributes=True)  # v2 写法
    id: int
    username: str

# db_user 是 SQLAlchemy 模型实例
user_out = UserOut.model_validate(db_user)  # 按属性名读取

#### 4. 模型配置

In [ ]:
from pydantic import BaseModel, ConfigDict

class User(BaseModel):
    model_config = ConfigDict(
        # 常用配置
        str_strip_whitespace=True,     # 自动去除字符串首尾空格
        str_to_lower=True,             # 字符串自动转小写
        str_to_upper=False,            # 字符串自动转大写
        frozen=True,                   # 模型不可变（类似 dataclass frozen）
        extra="forbid",                # 禁止额外字段 ("allow"/"ignore"/"forbid")
        populate_by_name=True,         # 允许通过字段名或别名赋值
        from_attributes=True,          # 支持从 ORM 对象创建（原 orm_mode）
        validate_assignment=True,      # 赋值时也进行校验
    )

    name: str
    age: int

字段别名

In [31]:
from pydantic import BaseModel, Field
class User(BaseModel):
    model_config = ConfigDict(populate_by_name=True)
    user_name: str = Field(alias="userName")    # 驼峰 → 下划线
    user_age: int = Field(alias="userAge")

# 通过别名创建（需开启 populate_by_name）
user = User(userName="Alice", userAge=25)

# 通过字段名创建
user2 = User(user_name="Bob", user_age=30)

# 输出时使用别名
print(user.model_dump(by_alias=True))
# {'userName': 'Alice', 'userAge': 25}

{'userName': 'Alice', 'userAge': 25}


#### 5. 常见内置类型

In [33]:
from pydantic import (
    EmailStr,       # 邮箱格式校验
    HttpUrl,        # HTTP/HTTPS URL
    AnyUrl,         # 任意 URL
    IPvAnyAddress,  # IP 地址
    UUID4,          # UUID v4
    SecretStr,      # 敏感字符串（打印时显示 **********）
    SecretBytes,    # 敏感字节
    FilePath,       # 必须存在的文件路径
    DirectoryPath,  # 必须存在的目录路径
    PositiveInt,    # 正整数
    NegativeFloat,  # 负浮点数
    conint,         # 约束整数（工厂函数）
    constr,         # 约束字符串（工厂函数）
)

from pydantic import BaseModel, SecretStr, PositiveInt

class Account(BaseModel):
    username: str
    password: SecretStr       # 打印时自动隐藏
    age: PositiveInt          # 必须 > 0

acc = Account(username="alice", password="123456", age=25)
print(acc.password)           # **********
print(acc.password.get_secret_value())  # 123456（获取真实值）

**********
123456


#### 6. 错误处理

In [32]:
from pydantic import BaseModel, ValidationError

class User(BaseModel):
    name: str
    age: int

try:
    User(name="Alice", age="abc")
except ValidationError as e:
    print(e.error_count())       # 错误数量
    print(e.errors())            # 错误列表（字典）
    print(e.json())              # JSON 格式错误信息

    for error in e.errors():
        print(f"字段: {error['loc']}, 错误: {error['msg']}, 输入: {error['input']}")

1
[{'type': 'int_parsing', 'loc': ('age',), 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'abc', 'url': 'https://errors.pydantic.dev/2.11/v/int_parsing'}]
[{"type":"int_parsing","loc":["age"],"msg":"Input should be a valid integer, unable to parse string as an integer","input":"abc","url":"https://errors.pydantic.dev/2.11/v/int_parsing"}]
字段: ('age',), 错误: Input should be a valid integer, unable to parse string as an integer, 输入: abc


#### 7. 动态模型创建


In [34]:
from pydantic import create_model

# 动态创建模型
DynamicUser = create_model(
    "DynamicUser",
    name=(str, ...),           # (类型, 默认值/...)
    age=(int, 0),
    email=(str, "default@example.com"),
)

user = DynamicUser(name="Alice")
print(user.age)    # 0
print(user.email)  # default@example.com


0
default@example.com
